🔹 Question 1: Load the Dataset (Easy) 

Create a SparkSession 

Load the CSV file into a Spark DataFrame 

Display the schema and first 5 records 

Expected Skills: 
spark.read.csv(), inferSchema, show(), printSchema() 

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import os, time  
os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"
spark = (
   SparkSession.builder
     .appName("PySpark Test")
     .master("local[*]")
    .getOrCreate()
)
 
sc = spark.sparkContext

In [13]:
csv_file = spark.read.csv("./sales_data.csv", header=True, inferSchema=True)

In [14]:
csv_file.show(5)

+--------+----------+-----------+----------+-----------+--------+-----+------------+----------+
|order_id|order_date|customer_id|   product|   category|quantity|price|payment_mode|     city |
+--------+----------+-----------+----------+-----------+--------+-----+------------+----------+
|     101|2024-01-01|       C001|    Laptop|Electronics|       1|70000| Credit Card|Bangalore |
|     102|2024-01-01|       C002|    Mobile|Electronics|       2|20000|         UPI|  Chennai |
|     103|2024-01-02|       C003|Headphones|Accessories|       3| 3000|  Debit Card|Hyderabad |
|     104|2024-01-02|       C001|  Keyboard|Accessories|       1| 1500| Credit Card|Bangalore |
|     105|2024-01-03|       C004|   Monitor|Electronics|       1|12000| Net Banking|     Pune |
+--------+----------+-----------+----------+-----------+--------+-----+------------+----------+
only showing top 5 rows



In [15]:
csv_file.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- city : string (nullable = true)



Question 2: Add a Derived Column (Easy) 

Create a new column called total_amount 

Formula: 

total_amount = quantity * price 

 Expected Skills: 
withColumn(), column expressions 


In [17]:
df = csv_file.withColumn("total_amount", col("quantity") * col("price"))

In [18]:
df.show(5)

+--------+----------+-----------+----------+-----------+--------+-----+------------+----------+------------+
|order_id|order_date|customer_id|   product|   category|quantity|price|payment_mode|     city |total_amount|
+--------+----------+-----------+----------+-----------+--------+-----+------------+----------+------------+
|     101|2024-01-01|       C001|    Laptop|Electronics|       1|70000| Credit Card|Bangalore |       70000|
|     102|2024-01-01|       C002|    Mobile|Electronics|       2|20000|         UPI|  Chennai |       40000|
|     103|2024-01-02|       C003|Headphones|Accessories|       3| 3000|  Debit Card|Hyderabad |        9000|
|     104|2024-01-02|       C001|  Keyboard|Accessories|       1| 1500| Credit Card|Bangalore |        1500|
|     105|2024-01-03|       C004|   Monitor|Electronics|       1|12000| Net Banking|     Pune |       12000|
+--------+----------+-----------+----------+-----------+--------+-----+------------+----------+------------+
only showing top 5 

🔹 Question 3: Filter Transactions (Easy–Moderate) 

Filter all transactions where: 

category is Electronics 

AND total_amount > 20,000 

Display the result 

 Expected Skills: 
filter(), logical conditions 

In [24]:
filtered_df = df.filter((col("total_amount") > 20000) & (col("category") == "Electronics"))


In [25]:
filtered_df.show()

+--------+----------+-----------+-------+-----------+--------+-----+------------+----------+------------+
|order_id|order_date|customer_id|product|   category|quantity|price|payment_mode|     city |total_amount|
+--------+----------+-----------+-------+-----------+--------+-----+------------+----------+------------+
|     101|2024-01-01|       C001| Laptop|Electronics|       1|70000| Credit Card|Bangalore |       70000|
|     102|2024-01-01|       C002| Mobile|Electronics|       2|20000|         UPI|  Chennai |       40000|
|     107|2024-01-04|       C005| Tablet|Electronics|       1|25000| Credit Card|    Delhi |       25000|
|     108|2024-01-04|       C003| Laptop|Electronics|       1|68000|  Debit Card|Hyderabad |       68000|
+--------+----------+-----------+-------+-----------+--------+-----+------------+----------+------------+



Question 4: Category-wise Revenue (Moderate) 

Calculate total revenue per category 

Output columns: 

category 

total_revenue 

Sort the result in descending order of revenue 

 Expected Skills: 
groupBy(), sum(), orderBy() 

Note:groupBy() and agg() uses task Optimization automatically.


In [26]:
from pyspark.sql.functions import sum

revenue_df = df.groupBy("category").agg(sum("total_amount").alias("total_revenue")).orderBy("total_revenue", ascending=False)

In [27]:
revenue_df.show()

+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|Electronics|       215000|
|Accessories|        12100|
+-----------+-------------+



Question 5: City-wise Order Count (Moderate) 

Find the number of orders per city 

Rename the count column to order_count 

 Expected Skills: 
groupBy(), count(), alias() 

In [32]:
from pyspark.sql.functions import count

orders_per_city = df.groupBy("city ").agg(count("order_id").alias("order_count"))


In [33]:
orders_per_city.show()

+----------+-----------+
|     city |order_count|
+----------+-----------+
|Hyderabad |          2|
|    Delhi |          1|
|  Chennai |          2|
|     Pune |          1|
|Bangalore |          2|
+----------+-----------+



 Question 6: Payment Mode Analysis (Moderate) 

Calculate total revenue per payment_mode 

Display only payment modes where revenue > 30,000 

 Expected Skills: 
Aggregation + filtering on aggregated results 


In [35]:
payment_mode = df.groupBy("payment_mode").agg(sum("total_amount").alias("total_revenue")).filter(col("total_revenue") > 30000)
payment_mode.show()

+------------+-------------+
|payment_mode|total_revenue|
+------------+-------------+
| Credit Card|        96500|
|  Debit Card|        77000|
|         UPI|        41600|
+------------+-------------+



Question 7: Highest Spending Customer (Moderate) 

Find the customer_id who spent the highest total amount 

Display: 

customer_id 

total_spent 

Expected Skills: 
Grouping, aggregation, sorting, limit(1) 


Question 8: Write Output to File (Easy) 

Write the category-wise revenue result to a CSV file 

Use: 

overwrite mode 

header = true 

 Expected Skills: 
write.mode().option().csv() 

 

 Spark Processing Flow (Visual Reference) 

A diagram of cluster manager

AI-generated content may be incorrect. 

A diagram of a spark diagram

AI-generated content may be incorrect. 

A diagram of a data frame

AI-generated content may be incorrect. 